
# Muon Counter Plotter — English Version

This notebook reads one or more plain-text files where each line has two comma-separated fields:

```
<muon_id>,<time_in_seconds>
```

It extracts:
- the **muon sequential number** (or cumulative count), and
- the **detection time** (seconds),

and then produces:
1. Individual time-vs-count plots for each location/dataset, and  
2. One comparison figure that overlays all datasets.

> **Customize the file paths below in the `Main` section.**


## Imports

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from typing import List, Tuple


## Helper functions (parsers & loaders)

In [ ]:

def parse_muon_number(line: str) -> int:
    """Return the muon sequential number (first CSV field)."""
    parts = line.strip().split(",")
    return int(parts[0])


def parse_muon_time(line: str) -> int:
    """Return the muon detection time in seconds (second CSV field)."""
    parts = line.strip().split(",")
    return int(parts[1])


def extract_data(lines: List[str]):
    """
    Given lines like 'N,T', return two lists:
      - n_muon: muon sequential numbers (int)
      - t_muon: times in seconds (int)
    Skips empty or malformed lines.
    """
    n_muon = []
    t_muon = []
    for line in lines:
        line = line.strip()
        if not line or "," not in line:
            continue
        try:
            n_muon.append(parse_muon_number(line))
            t_muon.append(parse_muon_time(line))
        except Exception:
            # Skip malformed lines silently
            continue
    return n_muon, t_muon


## Plot helpers

In [ ]:

def plot_muon_counts(x_time, y_count, color, label):
    """Plot a single time vs count curve."""
    plt.figure(figsize=(9, 4.5))
    plt.plot(x_time, y_count, color=color, label=label)
    plt.xlabel("Time (s)")
    plt.ylabel("Cumulative muon count")
    plt.title("Muon counts vs. time")
    plt.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


def plot_all_together(
    x0, y0, color0,
    x1, y1, color1,
    x2, y2, color2,
    labels=("Location A", "Location B", "Location C"),
    y_limit=None
):
    """Overlay three time vs count curves in a single figure for comparison."""
    plt.figure(figsize=(10, 5))
    plt.plot(x0, y0, color=color0, label=labels[0])
    plt.plot(x1, y1, color=color1, label=labels[1])
    plt.plot(x2, y2, color=color2, label=labels[2])
    plt.xlabel("Time (s)")
    plt.ylabel("Cumulative muon count")
    plt.title("Muon counts vs. time — comparison")
    if y_limit is not None:
        plt.ylim(0, y_limit)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


## Main — customize file paths and run

In [ ]:

# TODO: Update these paths to your data files.
# Each file must contain two comma-separated columns per line: '<muon_id>,<time_seconds>'
file_a = Path("/path/to/LocA_1850m.txt")
file_b = Path("/path/to/LocB_605m.txt")
file_c = Path("/path/to/LocC_580m.txt")

# Read files
lines_a = file_a.read_text(encoding="utf-8").splitlines()
lines_b = file_b.read_text(encoding="utf-8").splitlines()
lines_c = file_c.read_text(encoding="utf-8").splitlines()

# Extract data
n_muon_a, t_muon_a = extract_data(lines_a)
n_muon_b, t_muon_b = extract_data(lines_b)
n_muon_c, t_muon_c = extract_data(lines_c)

# Plot individual figures
plot_muon_counts(t_muon_a, n_muon_a, color="tab:red",    label="Location A — 1850 m")
plot_muon_counts(t_muon_b, n_muon_b, color="tab:cyan",   label="Location B — 605 m")
plot_muon_counts(t_muon_c, n_muon_c, color="tab:purple", label="Location C — 580 m")

# Plot comparison
plot_all_together(
    t_muon_a, n_muon_a, "tab:red",
    t_muon_b, n_muon_b, "tab:cyan",
    t_muon_c, n_muon_c, "tab:purple",
    labels=("Location A — 1850 m", "Location B — 605 m", "Location C — 580 m"),
    y_limit=None  # Set to e.g. 600 to force an upper bound
)
